# Matched-width learning curves and permutation controls

Figure 1, its stationary reference, Appendix C, and the step-size-switch control. Alternative layouts and duplicate width plots have been removed. Width curves remain in notebooks 01 and 05.


## Data and stationary reference

The networks have 24 hidden units each. The non-stationary problem has four subproblems and one label permutation every 800 steps. Each final curve uses 30 seeds.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    fetch_per_seed_arrays, get_color_palette, label_curves_inline, load_export, plot_ci_curve, set_style,
)

FIG_DIR = Path('../figures/generated')
OLD_DATA = Path('../../../phd/structure_search/analysis/weight_pruning/data')
PALETTE = get_color_palette(classes=['Block-Sparse', 'Dense'])

# Pre-release runs, one child run per seed (parent rows hold seed means).
params = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_params.csv',
                     index_col=0, low_memory=False)
params = params[params['child_seed'].notna()]
metrics = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_metrics.csv',
                      usecols=['run_id', 'step', 'loss', 'accuracy'])
metrics = metrics[metrics['loss'].notna()]   # rows without a loss are end-of-run summaries


def per_seed(sweep, units):
    """(step, seed) accuracy of one sweep at one width."""
    runs = params[(params['sweep_name'] == sweep) & (params['model.initial_hidden_units'] == units)]
    return (metrics[metrics['run_id'].isin(runs['run_id'])]
            .pivot(index='step', columns='run_id', values='accuracy'))


def last_tenth(acc):
    """Mean ± standard error over seeds of each seed's last-10% accuracy."""
    per_run = acc.iloc[-len(acc) // 10:].mean()
    return f'{per_run.mean():.4f} ± {per_run.std() / np.sqrt(len(per_run)):.4f}'


# Stationary ceilings: the 24-unit networks rerun for 2M steps (best/*_stationary_2m.yaml),
# since dense is still climbing at 500K. Each is the seed mean over the last 10% (1.8M-2M).
STATIONARY_2M = {'block_sparse_4task': '06_block_sparse_stationary_2m',
                 'dense_scaling_4task': '06_dense_stationary_2m'}
rebuttal_params, _ = load_export('paper-weight-pruning-rebuttal', data_dir='data')


def stationary_2m(stem):
    run_id = rebuttal_params.loc[rebuttal_params['sweep_name'] == STATIONARY_2M[stem], 'run_id'].item()
    accs = fetch_per_seed_arrays([run_id], cache_dir='data/per_seed_cache')[run_id]['accs']
    return accs[-len(accs) // 10:].mean()


for fmt in ('pdf', 'png'):
    (FIG_DIR / fmt).mkdir(parents=True, exist_ok=True)


### Stationary learning curves to 2M steps

The two networks above rerun for 2M steps with the same step-sizes and seeds, to see where each converges; 1000-step means with 95% bands. The first 500K steps should match the runs above. Needs the `paper-weight-pruning-rebuttal` export in `data/`.

In [ ]:
from multi_mnist.analysis import fetch_per_seed_arrays, load_export

# The same networks, step-sizes and seeds as above, run to 2M steps (best/*_stationary_2m.yaml).
rebuttal_params, _ = load_export('paper-weight-pruning-rebuttal', data_dir='data')
set_style('2-col')
fig, ax = plt.subplots()
specs = []
for label, stem, x, side, color in [
        ('Block-sparse', 'block_sparse', 0.4e6, 'above', PALETTE['Block-Sparse']),
        ('Dense', 'dense', 0.6e6, 'below', PALETTE['Dense'])]:
    run_id = rebuttal_params.loc[rebuttal_params['sweep_name'] == f'06_{stem}_stationary_2m', 'run_id'].item()
    accs = fetch_per_seed_arrays([run_id], cache_dir='data/per_seed_cache')[run_id]['accs']
    steps = (np.arange(len(accs)) + 1) * 1000
    old = per_seed(f'01_{stem}{"_scaling" if stem == "dense" else ""}_4task_best', 24)
    mean = accs.mean(axis=1)
    print(f'{label}: max |2M run - 500K run| over the first 500K steps (seed mean) '
          f'{np.abs(mean[:500] - old.mean(axis=1).to_numpy()).max():.4f}; last 10% '
          + ', '.join(f'to {end // 1000}K {mean[end // 1000 - end // 10_000:end // 1000].mean():.4f}'
                      for end in (500_000, 1_000_000, 1_500_000, 2_000_000)))
    plot_ci_curve(steps, accs, color=color, ax=ax)
    specs.append((label, x, side, color, (steps, mean)))
ax.set_xlim(0, 2_000_000); ax.set_ylim(0.4, 0.95)
ax.set_xlabel('Step'); ax.set_ylabel('Accuracy'); ax.set_title('Stationary, 2M Steps')
ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
fig.tight_layout()
label_curves_inline(ax, specs)
plt.show()

## Figure 1 — non-stationary learning and a view of six permutations

The dotted lines show final-10% accuracy from the 2-million-step stationary runs.


In [ ]:
log50 = rebuttal_params[rebuttal_params['sweep_name'] == '06_nonstationary_log50_last_step']
ceilings = [stationary_2m(stem) for stem in ('block_sparse_4task', 'dense_scaling_4task')]


In [ ]:
# Figure 1: the first 200K steps of the non-stationary runs in 50-step means (left, 3/5 of the
# width) and the last 6 permutations magnified (right, 2/5). Uses `log50` and `ceilings` from the reference cell.
END = 200_000
WIN = (END - 4_800, END)          # the last 6 permutations shown
set_style('2-col')
fig, (ax, zoom) = plt.subplots(1, 2, figsize=(14, 4.25), gridspec_kw=dict(width_ratios=[3, 2]))
specs = []
for (label, strategy, color, side, x_lab), ceiling in zip(
        [('Block-sparse', 'block_sparse', PALETTE['Block-Sparse'], 'above', 0.01e5),
         ('Dense', 'dense', PALETTE['Dense'], 'below', 0.45e5)], ceilings):
    run_id = log50.loc[log50['model.init_strategy'] == strategy, 'run_id'].item()
    accs = fetch_per_seed_arrays([run_id], cache_dir='data/per_seed_cache')[run_id]['accs']
    steps = (np.arange(len(accs)) + 1) * 50
    mean = accs.mean(axis=1)
    keep = steps <= END
    ax.plot(steps[keep], mean[keep], color=color, linewidth=1.0, alpha=0.75)
    specs.append((label, x_lab, side, color, (steps[keep], mean[keep])))
    keep = (steps > WIN[0]) & (steps <= WIN[1])
    plot_ci_curve(steps[keep], accs[keep], color=color, ax=zoom)
    for a in (ax, zoom):
        a.axhline(ceiling, linestyle=':', color=color, linewidth=3)
ax.set_xlim(0, END); ax.set_ylim(0.4, 0.97); ax.set_xticks(range(0, END + 1, 50_000))   # the zoomed panel's top too
ax.set_ylabel('Accuracy'); ax.set_title('Non-Stationary')
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
zoom.set_xlim(*WIN); zoom.set_ylim(0.6, 0.97); zoom.set_title('Last 6 Permutations')
zoom.set_xticks(range(196_000, WIN[1] + 1, 2000))
zoom.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True, useOffset=False)
zoom.set_yticks([0.6, 0.7, 0.8, 0.9]); zoom.yaxis.tick_right()   # ticks on the right keep the gap clear for the zoom lines
for spine in zoom.spines.values():   # framed, like the box it magnifies
    spine.set_visible(True)
for a in (ax, zoom):
    a.set_xlabel('Step'); a.grid(True, alpha=0.4); a.set_axisbelow(True)
fig.tight_layout()
fig.subplots_adjust(wspace=0.1)
# Box the magnified region and join its left corners to the zoomed panel's.
box, lines = ax.indicate_inset_zoom(zoom, facecolor=(0, 0, 0, 0.08), edgecolor='k', alpha=None,
                                    linewidth=plt.rcParams['axes.linewidth'])
box.set_clip_on(False)   # its right edge sits on the axes edge
for line, show in zip(lines, (True, True, False, False)):   # lower-left, upper-left, lower-right, upper-right
    line.set_visible(show); line.set_linewidth(plt.rcParams['axes.linewidth'])
texts = label_curves_inline(ax, specs)
texts[0].set_y(texts[0].get_position()[1] - 0.014)   # Block-sparse centred between its band and the dotted lines
label_curves_inline(zoom, [('Stationary accuracy', WIN[0] + 150, 'above', '0.3',
                            (np.linspace(*WIN, 400), np.full(400, max(ceilings))))])
for fmt in ('pdf', 'png'):
    fig.savefig(FIG_DIR / fmt / f'fig1_nonstationary_split.{fmt}', bbox_inches='tight', dpi=400)
plt.show()

## Appendix: block-sparse against dense across permutation intervals

Four-subproblem multi-MNIST with one subproblem's labels re-permuted every 200 to 6400 steps, block-sparse at 24 hidden units (19K weights) against dense at 24 (76K weights), matching Figure 1. Each point is the accuracy over the last 10% of training at that interval's best step-size, averaged over 5 seeds; each dotted line is that method's stationary accuracy over the last 10% of a 500K-step run (30 seeds). Permutation events inside a 1000-step logging window are all applied at the window's start, so intervals of 200 and 400 steps arrive as bursts of 5 and 2–3 events.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    fetch_per_seed_arrays, filter_sweep, get_color_palette, label_curves_inline,
    load_export, normalize_columns, plot_ci_curve, set_style, stack_per_seed,
)

PALETTE = get_color_palette(classes=['Block-Sparse', 'Dense'])
OLD_DATA = Path('../../../phd/structure_search/analysis/weight_pruning/data')  # pre-release MLflow export
N_PERIODS = 500   # 500,000 steps logged every 1000


def last_tenth(acc):
    """Mean accuracy over a run's last 10% of steps; NaN if it diverged and stopped early."""
    acc = acc.dropna()
    return acc.iloc[-N_PERIODS // 10:].mean() if len(acc) == N_PERIODS else np.nan


def best_per_period(params, metrics):
    """Each permutation interval's accuracy and step-size at its best step-size."""
    acc = (metrics[metrics['run_id'].isin(params['run_id'])]
           .groupby('run_id')['accuracy'].apply(last_tenth))
    cells = params.assign(acc=params['run_id'].map(acc),
                          period=pd.to_numeric(params['task.permute_period']))
    return cells.loc[cells.groupby('period')['acc'].idxmax()].set_index('period')


# Block-sparse, and dense at 800 steps, are pre-release sweeps (parent runs carry the
# 5-seed mean); dense at the other intervals is sweeps/01_matched_and_scaling/sweep/dense_permute_period.yaml.
old_params = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_sweep_params.csv',
                         index_col=0, low_memory=False)
old_params = old_params[old_params['child_seed'].isna()]
old_metrics = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_sweep_metrics.csv',
                          usecols=['run_id', 'step', 'accuracy'])
new_params, new_metrics = load_export('paper-weight-pruning-rebuttal', data_dir='data')
dense_800 = old_params[(old_params['sweep_name'] == '02_dense_scaling_4task_nonstationary_lr')
                       & (old_params['model.initial_hidden_units'] == 24)]
dense_rest = new_params[new_params['sweep_name'] == '07_dense_permute_period']

# Stationary limits: Figure 2a's 30-seed runs, one child run per seed (2^-10 and 2^-11).
best_params = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_params.csv',
                          index_col=0, low_memory=False)
best_metrics = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_metrics.csv',
                           usecols=['run_id', 'step', 'loss', 'accuracy'])
best_metrics = best_metrics[best_metrics['loss'].notna() & (best_metrics['step'] > 450_000)]


def stationary(sweep, units):
    runs = best_params[(best_params['sweep_name'] == sweep) & best_params['child_seed'].notna()
                       & (best_params['model.initial_hidden_units'] == units)]
    return best_metrics[best_metrics['run_id'].isin(runs['run_id'])]['accuracy'].mean()


PERMUTE = [  # (inline label, side[, parenthetical]), best per interval, stationary limit, color
    (('Block-sparse', 'above'),
     best_per_period(old_params[old_params['sweep_name'] == '04_block_sparse_permute_period_lr'],
                     old_metrics),
     stationary('01_block_sparse_4task_best', 24), PALETTE['Block-Sparse']),
    (('Dense', 'below'),
     pd.concat([best_per_period(dense_800, old_metrics),
                best_per_period(dense_rest, new_metrics)]).sort_index(),
     stationary('01_dense_scaling_4task_best', 24), PALETTE['Dense']),
]

set_style('2-col')
fig, ax = plt.subplots()
specs = []
for (text, side, *sub), best, limit, color in PERMUTE:
    print(f'{text}: stationary {limit:.4f}; best step-size and accuracy per interval\n'
          + best.assign(log2_step_size=np.log2(best['optimizer.learning_rate'].astype(float)))
          [['log2_step_size', 'acc']].round(4).to_string())
    ax.plot(best.index, best['acc'], '-o', color=color)
    ax.axhline(limit, linestyle=':', color=color)
    x = np.geomspace(best.index[0], best.index[-1], 400)   # the drawn line, for label clearance
    specs.append((text, 210, side, color, (x, np.interp(np.log2(x), np.log2(best.index), best['acc'])), *sub))

periods = PERMUTE[0][1].index
ax.set_xscale('log', base=2); ax.minorticks_off()
ax.set_xticks(periods); ax.set_xticklabels([f'{p:.0f}' for p in periods])
ax.set_xlim(periods[0] / 1.15, periods[-1] * 1.15); ax.set_ylim(0.5, 1.0)
ax.set_xlabel('Steps Between Permutations'); ax.set_ylabel('Accuracy')
ax.set_title('Permutation Interval')
ax.grid(True); ax.set_axisbelow(True)
fig.tight_layout()
label_curves_inline(ax, specs)
for fmt in ('pdf', 'png'):
    fig.savefig(f'../figures/generated/{fmt}/rev_appendix_permute_period.{fmt}',
                bbox_inches='tight', dpi=400)
plt.show()

## Dense transition with a step-size switch

This retained control changes the step-size from 2⁻⁸ to 2⁻¹¹ when dense connections appear. It uses 30 seeds and the same setup as Figure 2.


In [ ]:
FIG3_SWEEPS = ['03_dense_transition_n16_h256', '00_block_sparse_dense_fill_main_n16_h256',
               '00_block_sparse_dense_fill_main_n16_h256_lrext']
FIG3_CELLS = ['task.n_tasks', 'model.initial_hidden_units', 'algorithm.transition_step',
              'seed_offset', 'optimizer.learning_rate']
TRANSITION, NEVER = 250_000, 600_000   # a transition past the end of training is block-sparse

fig3_params, fig3_metrics = load_export('paper-weight-pruning-scaling-sweep', data_dir='data')
fig3_params, _ = filter_sweep(normalize_columns(fig3_params), fig3_metrics, FIG3_SWEEPS,
                              cell_cols=FIG3_CELLS)


def fig3_runs(transition_step, log2_step_size):
    ts = pd.to_numeric(fig3_params['algorithm.transition_step'], errors='coerce')
    lr = pd.to_numeric(fig3_params['optimizer.learning_rate'], errors='coerce')
    return fig3_params[(ts == transition_step)
                       & np.isclose(np.log2(lr), log2_step_size)]['run_id'].tolist()


# Comet ran two of the six seed blocks twice; the copies are identical.
switch_runs = (new_params[new_params['sweep_name'] == '05_dense_transition_step_size_drop_n16_h256']
               .drop_duplicates('seed_offset')['run_id'].tolist())

colors = get_color_palette()   # Figure 3's colors; slate (slot 5) is not used there
FIG3_CURVES = [  # (inline label, label x, side[, parenthetical]), run ids, color; drawing order
    (('Dense', 0.15e5, 'below'), fig3_runs(0, -11), colors[1]),
    (('Dense transition', 0.16e5, 'above'), fig3_runs(TRANSITION, -10), colors[3]),
    (('Block-sparse', 3.7e5, 'above'), fig3_runs(NEVER, -8), colors[0]),
    (('Dense transition', 2.62e5, 'above', r'($\to 2^{-11}$)'), switch_runs, colors[5]),
    (('Dense transition', 0.17e5, 'above', '(step-size = $2^{-8}$)'),
     fig3_runs(TRANSITION, -8), colors[2]),
]

set_style('2-col')
fig, ax = plt.subplots()
specs = []
for (text, x, side, *sub), run_ids, color in FIG3_CURVES:
    _, accs = stack_per_seed(fetch_per_seed_arrays(run_ids, cache_dir='data/per_seed_cache'),
                             run_ids)
    steps = np.arange(len(accs)) * 1000
    plot_ci_curve(steps, accs, color=color, ax=ax)
    last = accs[-N_PERIODS // 10:].mean(axis=0)
    print(f'{text} {sub[0] if sub else ""}: {accs.shape[1]} seeds, last 10% '
          + (f'{last.mean():.4f} ± {last.std(ddof=1) / np.sqrt(len(last)):.4f} (SE)'
             if len(accs) == N_PERIODS else f'none, diverged by step {len(accs) * 1000:,}'))
    mean = accs.mean(axis=1)
    keep = ~np.isnan(mean)            # the 2^-8 transition diverges and its curve stops
    if x > TRANSITION:                # a label right of the transition only has to clear the curve there
        keep &= steps > TRANSITION
    specs.append((text, x, side, color, (steps[keep], mean[keep]), *sub))

ax.axvline(TRANSITION, linestyle=':', color='black', alpha=0.6, linewidth=1)
ax.set_xlim(0, 500_000); ax.set_ylim(0, 1.0)
ax.set_xlabel('Step'); ax.set_ylabel('Accuracy')
ax.set_title('Learning Curves')
ax.grid(True); ax.set_axisbelow(True)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
fig.tight_layout()
label_curves_inline(ax, specs)
for fmt in ('pdf', 'png'):
    fig.savefig(f'../figures/generated/{fmt}/q3_step_size_switch.{fmt}', bbox_inches='tight', dpi=400)
plt.show()